# GATED MODEL

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error


import random

seed = 50

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================
# 1. data
# ==========================
data = np.load("..\data\hhg_dataset_1.5cycles.npz")

HHG_spec = data["HHG_spec"]
laser_param = data["y"]   # [E0, sinCEP, cosCEP]
# --- HHG---
X = HHG_spec[:, 400:3400:3]     # HHG spec
X = np.log1p(X)              #  log

# normalization
X = X / np.max(X, axis=1, keepdims=True)

# laser params
y = laser_param[:, 0:3]      # E0, sinCEP, cosCEP

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=seed
)

# StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

# tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

#train_loader = DataLoader(
#    TensorDataset(X_train, y_train),
#    batch_size=32,
#    shuffle=True
#)

g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
    generator=g
)

# ==========================
# 2. Attention + MLP model
# ==========================

class AttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Attention
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, input_dim),
            #nn.Softmax(dim=1)
            nn.Sigmoid() 
        )

        # MLP
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            #nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        attn = self.attention(x)
#        x = x * (1.0 + 0.5 * attn)
        x = x * attn
#        x = x * (1.0 + attn)
        return self.mlp(x)

model = AttentionMLP(input_dim=X_train.shape[1])

optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
criterion = nn.MSELoss()

# ==========================
# 3. training
# ==========================

for epoch in range(2000):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)

        # 
        loss = (
            1.0 * F.mse_loss(pred[:,0], yb[:,0]) +
            1.0 * F.mse_loss(pred[:,1], yb[:,1]) +
            1.0 * F.mse_loss(pred[:,2], yb[:,2])
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss {total_loss:.4f}")

# ==========================
# 4. eval
# ==========================

model.eval()
with torch.no_grad():
    pred_test = model(X_test)

# inversed transform
pred_test = scaler_y.inverse_transform(pred_test.numpy())
y_test_inv = scaler_y.inverse_transform(y_test.numpy())

# R2
print("=== R2 ===")
print("Intensity R2:", r2_score(y_test_inv[:,0], pred_test[:,0]))
print("sin CEP R2:", r2_score(y_test_inv[:,1], pred_test[:,1]))
print("cos CEP R2:", r2_score(y_test_inv[:,2], pred_test[:,2]))

# MAE
print("\n=== MAE (rad) ===")
print("sin:", mean_absolute_error(y_test_inv[:,1], pred_test[:,1]))
print("cos:", mean_absolute_error(y_test_inv[:,2], pred_test[:,2]))

cep_true = np.arctan2(y_test_inv[:,1], y_test_inv[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))
# 
# pi-shifted
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# comparison between 0 and pi-shift
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)



print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)

In [ ]:
cep_true = np.arctan2(y_test_inv[:,1], y_test_inv[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# comparison between 0 and pi-shift
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))

print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)

# save model

In [ ]:
torch.save(model.state_dict(), "../models/gatedMLP_seed50_n1.5cycle_lossMSE.pth")

# load model

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error


import random

seed = 50

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================
# 1. data
# ==========================
data = np.load("..\data\hhg_dataset_1.5cycles.npz")
#data = np.load("..\data\hhg_dataset_SFA_HHG_Laserparam_15cycles_CEP0-0.5piE0-0.08.npz")


HHG_spec = data["HHG_spec"]
laser_param = data["y"]   # [E0, sinCEP, cosCEP]
# --- HHG ---
X = HHG_spec[:, 400:3400:3]     # 
X = np.log1p(X)              # 

# normalization
X = X / np.max(X, axis=1, keepdims=True)

# laser param
y = laser_param[:, 0:3]      # E0, sinCEP, cosCEP

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=seed
)

# StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

# tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

model.load_state_dict(torch.load("../models/gatedMLP_seed50_n1.5cycle_lossMSE.pth"))
model.eval()   #


with torch.no_grad():
    pred_test = model(X_test)

# inversed transform
pred_test = scaler_y.inverse_transform(pred_test.numpy())
y_test_inv = scaler_y.inverse_transform(y_test.numpy())

# R2
print("=== R2 ===")
print("Intensity R2:", r2_score(y_test_inv[:,0], pred_test[:,0]))
print("sin CEP R2:", r2_score(y_test_inv[:,1], pred_test[:,1]))
print("cos CEP R2:", r2_score(y_test_inv[:,2], pred_test[:,2]))

# MAE
print("\n=== MAE (rad) ===")
print("sin:", mean_absolute_error(y_test_inv[:,1], pred_test[:,1]))
print("cos:", mean_absolute_error(y_test_inv[:,2], pred_test[:,2]))

cep_true = np.arctan2(y_test_inv[:,1], y_test_inv[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
# 
# pi-shifted
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# comparison between 0 and pi-shift
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))

print("MAE(rad) =", np.mean(np.abs(dcep1)))
print("MAE(deg) =", np.mean(np.abs(dcep1))*180/np.pi)
print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)

for i in range(20):
    print((dcep[i])*180/np.pi, cep_true[i]*180/np.pi, y_test_inv[i,0],pred_test[i,0])



# CEP retrieved error

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

import random

seed = 0

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================
# 1. data
# ==========================
data = np.load("..\hhg_dataset_1.5cycles.npz")
data2 = np.load("..\data\hhg_dataset_1.5cycles_CEP0-piE0-0.08.npz")


HHG_spec = data["HHG_spec"]
HHG_spec2=data2["HHG_spec"]
laser_param = data["y"]   # [E0, sinCEP, cosCEP]
laser_param2=data2["y"]
# --- HHG ---
X = HHG_spec[:, 400:3400:3]     #
X = np.log1p(X)              #
X2 = HHG_spec2[:, 400:3400:3]     #
X2 = np.log1p(X2)              #



# normalization
X = X / np.max(X, axis=1, keepdims=True)
X2 = X2/ np.max(X2, axis=1, keepdims=True)

# laser params
y = laser_param[:, 0:3]      # E0, sinCEP, cosCEP
y2= laser_param2[:, 0:3]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=seed
)

# StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)
X2=scaler_X.transform(X2)


scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

# tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
X2=torch.tensor(X2,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

model.load_state_dict(torch.load("../models/gatedMLP_seed0_n1.5cycle_lossMSE.pth"))
model.eval()   #


with torch.no_grad():
    pred_test = model(X2)

# inversed transform
pred_test = scaler_y.inverse_transform(pred_test.numpy())
#y_test_inv = scaler_y.inverse_transform(y_test.numpy())

# R2
#print("=== R2 ===")
#print("Intensity R2:", r2_score(y_test_inv[:,0], pred_test[:,0]))
#print("sin CEP R2:", r2_score(y_test_inv[:,1], pred_test[:,1]))
#print("cos CEP R2:", r2_score(y_test_inv[:,2], pred_test[:,2]))

# MAE
#print("\n=== MAE (rad) ===")
#print("sin:", mean_absolute_error(y_test_inv[:,1], pred_test[:,1]))
#print("cos:", mean_absolute_error(y_test_inv[:,2], pred_test[:,2]))

cep_true = np.arctan2(y2[:,1], y2[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
# 
# PI-SHIFTED
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# 
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))

print("MAE(rad) =", np.mean(np.abs(dcep1)))
print("MAE(deg) =", np.mean(np.abs(dcep1))*180/np.pi)
print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)

#for i in range(20):
print(laser_param2[50],pred_test[50], cep_true[50]*180/np.pi,cep_pred1[50]/np.pi)
data = np.column_stack([
    laser_param2,
    pred_test,
    cep_true * 180/np.pi,
    cep_pred1 * 180/np.pi
])

np.savetxt(
    "HHG_spec_E008_CEP0-pi.txt",
    data
)
plt.figure(figsize=(8,5))

plt.plot(
    cep_true*180/np.pi, abs(dcep*180/np.pi)
)

#plt.xticks(np.arange(0, 180, 20)) 
plt.xlim(0, 180)

#plt.ylim(1e-12, 1)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.xlabel("Input CEP (deg.)", fontsize=18)
plt.ylabel("Error (deg.)", fontsize=18)
#plt.title("waveform")
#
plt.grid(True)
plt.tight_layout()
#plt.savefig("waveformE0_01_CEP0_05pi.jpg", dpi=300, bbox_inches="tight")

plt.savefig("HHGE008_error.pdf", dpi=600, bbox_inches="tight")
plt.show()
